# 🎓 Continuous Internal Evaluation 2 (CIE-2)
## Activity No. 1: Dataset Preparation, Exploratory Data Analysis & Data Preprocessing

| **Attribute** | **Details** |
| :--- | :--- |
| **Activity** | Activity No. 1 |
| **Domain** | Tabular Data Cleaning, Preprocessing & EDA |
| **Dataset** | Student Academic Performance Dataset |
| **Target** | `Final_Exam_Score` |

---
### 📌 Objectives:
1. Ingest raw dataset and audit schema, column data types, and dimensions.
2. Identify and quantify data quality anomalies (missing values, whitespace discrepancies, exact duplicates).
3. Perform Exploratory Data Analysis (EDA): distribution analysis, frequency counts, outlier detection using $1.5 \times \text{IQR}$.
4. Implement data preprocessing pipeline: string trimming, deduplication, median/mode imputation, outlier capping, One-Hot Encoding, and `StandardScaler` normalization.
5. Export cleaned, analysis-ready dataset to `student_performance_prepared.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Set plot style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (8, 5)

# Load raw dataset
df = pd.read_csv("student_performance_dataset.csv")
original_df = df.copy()

print("Dataset shape:", df.shape)
print("\n--- First 5 Rows ---")
display(df.head())
print("\n--- Last 5 Rows ---")
display(df.tail())

In [ ]:
df.info()
print("\nColumn names:", df.columns.tolist())
print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)
print("\n--- Numerical Feature Statistics ---")
display(df[num_cols].describe().T)
print("\n--- Categorical Feature Statistics ---")
display(df[cat_cols].describe().T)

In [ ]:
missing_report = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage (%)": (df.isnull().sum() / len(df) * 100).round(2)
})
print("--- Missing Values Audit ---")
display(missing_report[missing_report["Missing Count"] > 0])
print("\nExact duplicate rows detected:", df.duplicated().sum())

In [ ]:
data = original_df.copy()

# Trim leading/trailing whitespace in categorical values
for col in data.select_dtypes(include="object").columns:
    data[col] = data[col].astype(str).str.strip()

# Impute numerical features using Median
for col in data.select_dtypes(include=np.number).columns:
    if data[col].isna().any():
        median_val = data[col].median()
        data[col] = data[col].fillna(median_val)
        print(f"Imputed missing in '{col}' with Median: {median_val:.2f}")

# Impute categorical features using Mode
for col in data.select_dtypes(include="object").columns:
    if data[col].isna().any():
        mode_val = data[col].mode()[0]
        data[col] = data[col].fillna(mode_val)
        print(f"Imputed missing in '{col}' with Mode: '{mode_val}'")

# Deduplication
before_duplicates = len(data)
data = data.drop_duplicates().copy()
print(f"\nRows removed as exact duplicates: {before_duplicates - len(data)}")
print("Remaining missing values:", int(data.isna().sum().sum()))
print("Remaining duplicate rows:", int(data.duplicated().sum()))

In [ ]:
print("--- Categorical Feature Value Distributions ---")
for col in ["Gender", "Parental_Education", "Extracurricular_Activities", "Internet_Access"]:
    print(f"\nValue counts for: {col}")
    display(data[col].value_counts().to_frame("Count"))

In [ ]:
print("--- Key Continuous Variables Summary ---")
key_numeric = ["Study_Hours_Per_Week", "Attendance_Rate", "Previous_Exam_Score", "Final_Exam_Score"]
display(data[key_numeric].describe().T)

In [ ]:
# Exploratory Data Analysis Plots
plt.figure(figsize=(8, 4))
sns.histplot(data=data, x="Final_Exam_Score", bins=20, kde=True, color="#2b5c8f")
plt.title("Distribution of Final Exam Scores (Target)")
plt.xlabel("Final Exam Score (0–100)")
plt.ylabel("Student Count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.countplot(data=data, x="Parental_Education", palette="viridis",
              order=["High School", "Associate Degree", "Bachelor's", "Master's", "Doctorate"])
plt.title("Distribution of Parental Education Levels")
plt.xlabel("Parental Education Level")
plt.ylabel("Number of Students")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.boxplot(data=data, x="Study_Hours_Per_Week", color="#81c784")
plt.title("Boxplot: Study Hours Per Week (Outlier Inspection)")
plt.xlabel("Weekly Study Hours")
plt.tight_layout()
plt.show()

# Outlier Capping (Winsorization) using 1.5 * IQR
q1 = data["Study_Hours_Per_Week"].quantile(0.25)
q3 = data["Study_Hours_Per_Week"].quantile(0.75)
iqr = q3 - q1
upper_cap = q3 + 1.5 * iqr
lower_cap = max(0, q1 - 1.5 * iqr)
data["Study_Hours_Per_Week"] = data["Study_Hours_Per_Week"].clip(lower=lower_cap, upper=upper_cap)
print(f"Capped 'Study_Hours_Per_Week' between [{lower_cap:.2f}, {upper_cap:.2f}]")

In [ ]:
# Drop non-predictive ID column
if "Student_ID" in data.columns:
    data = data.drop(columns=["Student_ID"])

# One-Hot Encoding
categorical_cols = data.select_dtypes(include="object").columns.tolist()
encoded = pd.get_dummies(data, columns=categorical_cols, drop_first=True, dtype=int)
print("Shape before encoding:", data.shape)
print("Shape after encoding:", encoded.shape)
display(encoded.head())

In [ ]:
# Feature Scaling using StandardScaler
scale_cols = ["Age", "Study_Hours_Per_Week", "Attendance_Rate", "Tutoring_Sessions", "Sleep_Hours_Per_Night", "Previous_Exam_Score"]
scale_cols = [c for c in scale_cols if c in encoded.columns]

scaler = StandardScaler()
encoded[scale_cols] = scaler.fit_transform(encoded[scale_cols])
print("Scaled continuous columns:", scale_cols)
display(encoded[scale_cols].agg(["mean", "std"]).round(4))

In [ ]:
print("Final Prepared Shape:", encoded.shape)
print("Remaining missing values:", int(encoded.isna().sum().sum()))
print("Remaining duplicate rows:", int(encoded.duplicated().sum()))
display(encoded.head())

# Save prepared output
encoded.to_csv("student_performance_prepared.csv", index=False)
print("🎉 Successfully saved: student_performance_prepared.csv")